# نَبْض | توليد شرح بصوت أنثوي بغدادي — Gemini 3.1 Flash TTS

هذا الدفتر يُنشئ **مسودة نص شرح من الموضوع والمعلومات العلمية التي تضعها**، ثم يحوّل النص الذي توافق عليه إلى ملف صوتي بصوت **Sulafat** الأنثوي الدافئ.

- نموذج الصوت: `gemini-3.1-flash-tts-preview` (اختيارك، وهو إصدار معاينة قديم).
- اللهجة المطلوبة: عراقية بغدادية، طبيعية وغير مبالغ بها.
- لا يحتاج اشتراكاً مدفوعاً ضمن حصة Google المجانية المتاحة لحسابك؛ قد تتغير الحصص أو تتوقف عند بلوغها.
- ستحتاج مفتاحاً مجانياً خاصاً من [Google AI Studio](https://aistudio.google.com/apikey). **لا تضعه داخل GitHub أو في التطبيق العام.**
- **الصوت الجديد مو نسخة من B**، لأن هذا النموذج لا يستنسخ الصوت المرجعي. يلزم الاستماع والموافقة قبل استخدام أي مقطع مع الطلاب.
- هذا اختبار **لموضوع واحد**، وليس إثباتاً أن الكتاب والملزمة صارا صوتاً. أضف نصاً متحققاً من المصدر قبل اعتماد أي شرح.


In [ ]:
%pip -q install --upgrade google-genai
from google import genai
from google.genai import types
from google.colab import userdata, files
from IPython.display import Audio, display
from getpass import getpass
from pathlib import Path
import json, re, wave, os

try:
    KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    KEY = None
if not KEY:
    KEY = getpass("الصق مفتاح Google AI Studio هنا، لن يظهر على الشاشة: ")
if not KEY or not KEY.strip():
    raise ValueError("لازم مفتاح Google API حتى يتم إنشاء الصوت.")
client = genai.Client(api_key=KEY.strip())
OUT = Path("/content/NABD_Gemini31_Baghdad")
OUT.mkdir(exist_ok=True)
print("جاهز لتجربة موضوع واحد. المفتاح لم يُحفظ داخل الملفات.")


## ١ — حدّد الموضوع والمعلومات المراجعة

غيّر المتغيرين أدناه إلى أي موضوع من الكتاب أو الملزمة، والصق نقاطاً علمية دقيقة من المصدر. لا تعتمد على اسم الموضوع وحده في مادة دراسية.

In [ ]:
TOPIC = "الخلية العصبية"
SOURCE_FACTS = """الخلية العصبية هي الوحدة التركيبية والوظيفية الأساسية للجهاز العصبي.
تتألف من جسم الخلية والتشجرات والمحور والنهايات العصبية.
تستقبل التشجرات الإشارات العصبية، وينقل المحور الإيعاز العصبي باتجاه النهايات العصبية."""
SCRIPT_OVERRIDE = ""  # إذا عندك النص النهائي المدقق، اكتبه هنا بين علامتي الاقتباس لتجاوز إنشاء المسودة.

if SCRIPT_OVERRIDE.strip():
    SCRIPT = SCRIPT_OVERRIDE.strip()
else:
    prompt = f"""اكتب مسودة شرح صوتي تعليمي لطالب الثالث المتوسط في العراق.
الموضوع: {TOPIC}
المصدر العلمي الملزم: {SOURCE_FACTS}
اللغة: لهجة عراقية بغدادية طبيعية من معلمة، وليست خليجية أو شامية أو فصحى رسمية.
الأسلوب: دافئ ومطمئن ومباشر، مناسب لعمر 13-16 سنة.
المطلوب: 65 إلى 95 كلمة تقريباً، جمل قصيرة، مثال أو تشبيه واحد فقط إن كان دقيقاً،
وسؤال ختامي للتأكد من الفهم. التزم فقط بالمعلومات المذكورة في المصدر.
لا تختلق معلومة وزارية ولا سنة امتحان ولا حقيقة طبية غير واردة بالمصدر.
لا تكتب عناوين أو قوائم أو علامات إخراج أو تنبيهات؛ أرجع النص المنطوق فقط."""
    draft = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.25)
    )
    SCRIPT = (draft.text or "").strip()
if not SCRIPT:
    raise RuntimeError("ما انكتب النص. تحقق من الحصة المجانية أو أدخل SCRIPT_OVERRIDE.")
(OUT/"lesson_draft.txt").write_text(SCRIPT, encoding="utf-8")
print("الموضوع:", TOPIC)
print("\nنص الشرح للمراجعة:\n")
print(SCRIPT)


## ٢ — الموافقة على النص قبل توليد الصوت

اقرأ النص، ودقّق المصطلحات والأسماء العلمية مقابل الكتاب/الملزمة. إذا مو صحيح، غيّر المصدر أو النص وأعد تشغيل الخلية السابقة. **اكتب نعم فقط إذا النص صحيح.**

In [ ]:
decision = input("هل النص صحيح وجاهز لتجربة النطق؟ اكتب نعم: ").strip()
if decision != "نعم":
    raise RuntimeError("وقف التوليد إلى أن تراجع النص. ما صار إنشاء صوت.")
print("النص معتمد لتجربة الصوت فقط، وليس للنشر التلقائي.")


## ٣ — توليد صوت المعلمة البغدادية

هذه الخلية تطلب من Gemini 3.1 نطق النص بصوت **Sulafat** الأنثوي. استمع للمقطع قبل اعتماده؛ اللهجة الدقيقة واستمرارية الهوية ليست مضمونة آلياً.

In [ ]:
AUDIO_MODEL = "gemini-3.1-flash-tts-preview"
VOICE_NAME = "Sulafat"
PERFORMANCE = """Synthesize one speaker reading an educational script, never a dialogue.
Audio profile: an adult female science teacher from Baghdad, Iraq.
Acting direction: warm, welcoming, affectionate but not childish, clear and patient;
a natural contemporary Baghdadi Iraqi Arabic accent, not Egyptian, Gulf, Levantine,
or formal newscaster Arabic. Keep the scientific terminology precise and carefully pronounced.
Speak at a comfortable unhurried teaching pace with natural pauses.
No music, background audio, or sound effects. Read only the text between TRANSCRIPT markers.
TRANSCRIPT:
"""
tts_response = client.models.generate_content(
    model=AUDIO_MODEL,
    contents=PERFORMANCE + SCRIPT + "\nEND TRANSCRIPT.",
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=VOICE_NAME)
            )
        )
    )
)
parts = tts_response.candidates[0].content.parts if tts_response.candidates else []
audio_part = next((p.inline_data for p in parts if getattr(p, "inline_data", None) and
                   (p.inline_data.mime_type or "").lower().startswith("audio/")), None)
if audio_part is None or not audio_part.data:
    raise RuntimeError("النموذج ما رجّع بيانات صوتية. لا تعتبر التجربة ناجحة.")
pcm_or_wav = audio_part.data
if isinstance(pcm_or_wav, str):
    import base64
    pcm_or_wav = base64.b64decode(pcm_or_wav)
wav_file = OUT / "neuron_baghdad_sulafat_draft.wav"
if pcm_or_wav[:4] == b"RIFF":
    wav_file.write_bytes(pcm_or_wav)
else:
    mime = audio_part.mime_type or ""
    rate = int(re.search(r"rate=(\d+)", mime).group(1)) if re.search(r"rate=(\d+)", mime) else 24000
    with wave.open(str(wav_file), "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(rate)
        w.writeframes(pcm_or_wav)
info = {
  "topic": TOPIC,
  "script_file": "lesson_draft.txt",
  "audio_file": wav_file.name,
  "tts_model": AUDIO_MODEL,
  "voice_name": VOICE_NAME,
  "accent_requested": "Baghdadi Iraqi Arabic",
  "audio_approved_for_students": False,
  "reason": "Generated test only; teacher must verify pronunciation, identity and facts."
}
(OUT/"lesson_metadata.json").write_text(json.dumps(info,ensure_ascii=False,indent=2), encoding="utf-8")
display(Audio(filename=str(wav_file)))
print("تم إنشاء ملف WAV تجريبي:", wav_file)
print("مهم: لم نعتبره صوت B ولم ننشره في التطبيق.")


## ٤ — تنزيل التسجيل بعد الاستماع

افحص اللهجة والبغدادية ونطق المصطلحات، ثم نزّل الملف. الاستيراد إلى التطبيق يجب أن يكون بصفة **«صوت Gemini»** مستقل عن هوية B.

In [ ]:
files.download(str(wav_file))
